# Data Wrangling Forecasting - Tesis

Checklist de calidad previo a entrenamiento:
- Precision: outliers, escala, duplicados, tipo/signo
- Completitud: ausentes

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'wrangling.py').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'wrangling.py').exists():
    raise RuntimeError('No se encontro src/wrangling.py. Abre el notebook dentro de 03_modelado/tesis_forecasting.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.dataset import load_monthly_production_dataset
from src.wrangling import prepare_modeling_table
from src.quality import build_quality_report, compare_quality_reports

In [2]:
raw = load_monthly_production_dataset(source='dwh', fallback_quickbooks=True)
print('RAW rows=', len(raw), 'productos=', raw['producto'].nunique(), 'periodos=', raw['periodo'].nunique())
display(raw.head())

RAW rows= 12381 productos= 1002 periodos= 21


F:\proyecto-integrador\Avance 2\03_modelado\tesis_forecasting\src\postgres_loader.py:67: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, conn)


,producto,anio,mes,qty_fabricada,qty_planificada,n_ordenes,periodo
0,1 CONDIMENSA > ACEITES Y GRASAS > ACEITE DE GI...,2024,7,6960.0,8110.0,5,2024-07-01
1,1 CONDIMENSA > ACEITES Y GRASAS > ACEITE DE GI...,2024,8,5440.0,7840.0,2,2024-08-01
2,1 CONDIMENSA > ACEITES Y GRASAS > ACEITE DE GI...,2024,9,6426.0,6811.0,7,2024-09-01
3,1 CONDIMENSA > ACEITES Y GRASAS > ACEITE DE GI...,2024,10,8439.0,10080.0,11,2024-10-01
4,1 CONDIMENSA > ACEITES Y GRASAS > ACEITE DE GI...,2024,11,7770.0,8720.0,8,2024-11-01


## Calidad de datos - Antes de wrangling

In [3]:
raw_quality = build_quality_report(raw, stage='raw')
display(raw_quality)

,stage,dimension,regla,count,pct_rows,status,rows_total
0,raw,Completitud,ausentes_columnas_criticas,0,0.000000,OK,12381
1,raw,Precision,outliers_iqr_qty_fabricada,1426,11.517648,REVISAR,12381
2,raw,Precision,outliers_iqr_qty_planificada,1432,11.566109,REVISAR,12381
3,raw,Precision,inconsistencia_escala_ratio_fab_plan,11,0.088846,REVISAR,12381
4,raw,Precision,inconsistencia_escala_salto_fabricada,147,1.187303,REVISAR,12381
5,raw,Precision,duplicados_producto_periodo,0,0.000000,OK,12381
6,raw,Precision,inconsistencia_tipo_qty_fabricada,0,0.000000,OK,12381
7,raw,Precision,inconsistencia_tipo_qty_planificada,0,0.000000,OK,12381
8,raw,Precision,inconsistencia_signo_qty_fabricada_neg,0,0.000000,OK,12381
9,raw,Precision,inconsistencia_signo_qty_planificada_neg,0,0.000000,OK,12381


## Wrangling para modelado

In [4]:
wrangled, wrangling_report = prepare_modeling_table(raw, min_periods_product=4)
print('WRANGLED rows=', len(wrangled), 'productos=', wrangled['producto'].nunique(), 'periodos=', wrangled['periodo'].nunique())
display(wrangling_report)
display(wrangled.head())

WRANGLED rows= 14670 productos= 904 periodos= 21


,metric,value
0,rows_output,14670
1,products_output,904
2,periods_output,21
3,rows_imputed_missing_month,2434
4,rows_qty_fabricada_clipped,707
5,rows_qty_planificada_clipped,713
6,imputation_mode,zero


,periodo,producto,qty_fabricada,qty_planificada,n_ordenes,imputado_mes_faltante
0,2024-07-01,1 CONDIMENSA ACEITES Y GRASAS ACEITE DE GIRASO...,6960.0,8110.0,5.0,False
1,2024-08-01,1 CONDIMENSA ACEITES Y GRASAS ACEITE DE GIRASO...,5440.0,7840.0,2.0,False
2,2024-09-01,1 CONDIMENSA ACEITES Y GRASAS ACEITE DE GIRASO...,6426.0,6811.0,7.0,False
3,2024-10-01,1 CONDIMENSA ACEITES Y GRASAS ACEITE DE GIRASO...,8439.0,10080.0,11.0,False
4,2024-11-01,1 CONDIMENSA ACEITES Y GRASAS ACEITE DE GIRASO...,7770.0,8720.0,8.0,False


## Calidad de datos - Despues de wrangling

In [5]:
wrangled_quality = build_quality_report(wrangled, stage='wrangled')
display(wrangled_quality)

,stage,dimension,regla,count,pct_rows,status,rows_total
0,wrangled,Completitud,ausentes_columnas_criticas,0,0.000000,OK,14670
1,wrangled,Precision,outliers_iqr_qty_fabricada,1672,11.397410,REVISAR,14670
2,wrangled,Precision,outliers_iqr_qty_planificada,1675,11.417860,REVISAR,14670
3,wrangled,Precision,inconsistencia_escala_ratio_fab_plan,12,0.081800,REVISAR,14670
4,wrangled,Precision,inconsistencia_escala_salto_fabricada,1512,10.306748,REVISAR,14670
5,wrangled,Precision,duplicados_producto_periodo,0,0.000000,OK,14670
6,wrangled,Precision,inconsistencia_tipo_qty_fabricada,0,0.000000,OK,14670
7,wrangled,Precision,inconsistencia_tipo_qty_planificada,0,0.000000,OK,14670
8,wrangled,Precision,inconsistencia_signo_qty_fabricada_neg,0,0.000000,OK,14670
9,wrangled,Precision,inconsistencia_signo_qty_planificada_neg,0,0.000000,OK,14670


## Comparacion antes vs despues

In [6]:
quality_cmp = compare_quality_reports(raw_quality, wrangled_quality)
display(quality_cmp)

,regla,count_raw,count_wrangled,delta,mejora
0,inconsistencia_escala_ratio_fab_plan,11,12,1,EMPEORA
1,inconsistencia_escala_salto_fabricada,147,1512,1365,EMPEORA
2,outliers_iqr_qty_fabricada,1426,1672,246,EMPEORA
3,outliers_iqr_qty_planificada,1432,1675,243,EMPEORA
4,ausentes_columnas_criticas,0,0,0,IGUAL
5,duplicados_producto_periodo,0,0,0,IGUAL
6,inconsistencia_signo_qty_fabricada_neg,0,0,0,IGUAL
7,inconsistencia_signo_qty_planificada_neg,0,0,0,IGUAL
8,inconsistencia_tipo_qty_fabricada,0,0,0,IGUAL
9,inconsistencia_tipo_qty_planificada,0,0,0,IGUAL


In [7]:
artifacts = ROOT / 'artifacts'
artifacts.mkdir(exist_ok=True)
wrangled.to_csv(artifacts / 'dataset_wrangled_forecasting.csv', index=False)
wrangling_report.to_csv(artifacts / 'wrangling_report.csv', index=False)
raw_quality.to_csv(artifacts / 'quality_report_raw.csv', index=False)
wrangled_quality.to_csv(artifacts / 'quality_report_wrangled.csv', index=False)
quality_cmp.to_csv(artifacts / 'quality_report_comparison.csv', index=False)
print('Artefactos de calidad guardados en', artifacts)

Artefactos de calidad guardados en F:\proyecto-integrador\Avance 2\03_modelado\tesis_forecasting\artifacts
